In [1]:
### IMPORTS
import os
import base64
from PIL import Image
from io import BytesIO
from openai import OpenAI
from dotenv import load_dotenv
from pdf2image import convert_from_path

In [2]:
load_dotenv()
OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY)

In [3]:
def convert_and_encode_pdf(pdf_path, output_format="PNG"):
    images = convert_from_path(pdf_path)

    encoded_images = []
    for image in images:
        buffered = BytesIO()
        image.save(buffered, format=output_format)
        encoded_image = base64.b64encode(buffered.getvalue()).decode('utf-8')
        encoded_images.append(encoded_image)
    
    return encoded_images

def check_image_size(encoded_image, max_size_mb=20):
    img_bytes = base64.b64decode(encoded_image)
    size_mb = len(img_bytes) / (1024 * 1024)
    return size_mb <= max_size_mb

def verify_image(encoded_image):
    image_data = base64.b64decode(encoded_image)
    image = Image.open(BytesIO(image_data))
    image.show()

In [5]:
# convert and encode the images
file_path = '/Users/hamiltones/Library/CloudStorage/OneDrive-Personal/Journal/Daily Pages/2024/10-2024/10-23-2024 AM.pdf'
b64_imgs = convert_and_encode_pdf(file_path)

# check that images are within the size limit
for encoded_image in b64_imgs:
    if check_image_size(encoded_image):
        print("Image within size limit")
        print(type(encoded_image))
        print(encoded_image[:500])
        verify_image(encoded_image)
    else:
        print("Image exceeds size limit")

Image within size limit
<class 'str'>
iVBORw0KGgoAAAANSUhEUgAABiEAAAfQCAIAAAArWmtJAAEAAElEQVR4nOz9ebQmWXIfhsVy7838trdUVVd1d1V3Y7qnZ8FwgCEAQeIAhECKoimuICVZFEnTomVSEEXJ9qElUYupY0syvUjHxzyiaZmUZFI7SHERB6DpAxGiIAoksQxmMINZMNPTPb3W8tZvy8x7I8J/RGa+r15V99TMgIuFF6dP9Xvfyy/z5l0jfvGLCPz3/v1/+95b76zfufeZT34yGBZcStV2XRO4xnZCbWDsBI4k6GkL1/avlePzZCEQXTu8ca761376KxDgxq3rr792VDF953d89+/5wd8nCJ/4xCdeffXV3/t7f+90Op3NZjlneAIppUwmk7ZtQwgnJyd379790Ic+tNmsXnvttXfeecfMQggAYGaPfvexH4pICIGZc84HBwfPPffcZz/7WVV96aWXXn311Q9+8IPXr19/7bXXnnnmmbquVdXMUkoPHjw4Ozt7


In [6]:
# send images to OpenAI
for image in b64_imgs:
    # decoded_img = base64.b64decode(image)
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                'role': 'user',
                'content': [
                    {
                        'type': 'text',
                        'text': 'Please transcribe this document.'
                    },
                    {
                        'type': 'image_url',
                        'image_url': {
                            'url': f"data:image/png;base64,{image}"
                        }
                    }
                ]
            }
        ]
    )

    print(response.choices)

Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="10/23/24 AM\n\nI'm feeling a bit down today and yesterday. I've been getting a lot of sleep but I also don't feel like I'm rested and ready to get to work. I'm stressed about my trip to SF, not unmanageably stressed but I'm still dreading it. I requested an accommodation at work to not be required to travel outside the DC area which was granted. So I'm grateful for that. Right now I just don't want to travel anywhere or meet anyone new. I'm also feeling stressed about money because it's flying out of my bank account about as quickly as it comes in. There are just so many bills coming for me and I need to pay off credit card debt from the NY trip in August. I'm also no longer on my parents' insurance so can't afford therapy anymore. It costs $150 an appointment now and I'm worried I'll be retroactively charged for that rate because my parent's insurance stopped covering me in August. So right now

In [10]:
print(response.choices[0].message.content)

10/23/24 AM

I'm feeling a bit down today and yesterday. I've been getting a lot of sleep but I also don't feel like I'm rested and ready to get to work. I'm stressed about my trip to SF, not unmanageably stressed but I'm still dreading it. I requested an accommodation at work to not be required to travel outside the DC area which was granted. So I'm grateful for that. Right now I just don't want to travel anywhere or meet anyone new. I'm also feeling stressed about money because it's flying out of my bank account about as quickly as it comes in. There are just so many bills coming for me and I need to pay off credit card debt from the NY trip in August. I'm also no longer on my parents' insurance so can't afford therapy anymore. It costs $150 an appointment now and I'm worried I'll be retroactively charged for that rate because my parent's insurance stopped covering me in August. So right now the vibe is that I've overshot my means a bit and I need to tighten up. I can do that especia